<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Leo/NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# =========================
# 0) Install (falls nötig)
# =========================
!pip -q install transformers datasets evaluate accelerate scikit-learn pandas

import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

In [14]:
# =========================
# 1) Load data
# =========================
DATA_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/Leo/department-v2.csv"
df = pd.read_csv(DATA_PATH)

# Basic cleanup
df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

# Optional: remove super-short texts that often add noise
df = df[df["text"].str.len() >= 2].copy()


In [15]:
# =========================
# 2) Label encoding
# =========================
labels = sorted(df["label"].unique().tolist())
label2id = {label_value: index for index, label_value in enumerate(labels)}
id2label = {index: label_value for label_value, index in label2id.items()}
df["label_id"] = df["label"].map(label2id).astype(int)

# Stratified split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label_id"]
)

train_ds = Dataset.from_pandas(train_df[["text", "label_id"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label_id"]], preserve_index=False)


In [16]:
# =========================
# 3) Tokenizer + tokenize fn
# =========================
MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=32)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label_id", "labels")
val_ds   = val_ds.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/8623 [00:00<?, ? examples/s]

Map:   0%|          | 0/1522 [00:00<?, ? examples/s]

In [17]:
# =========================
# 4) Model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
# =========================
# 5) Metrics
# =========================
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=y_pred, references=y_true)["accuracy"],
        "f1_macro": f1.compute(predictions=y_pred, references=y_true, average="macro")["f1"],
        "f1_weighted": f1.compute(predictions=y_pred, references=y_true, average="weighted")["f1"],
    }


In [20]:
# =========================
# 6) TrainingArguments + Trainer
# =========================
import torch

args = TrainingArguments(
    output_dir="./dept_model_runs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,

    weight_decay=0.01,
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)


/tmp/ipython-input-4209090681.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.504400,0.092231,0.980289,0.675543,0.976474
2,0.080400,0.056472,0.987516,0.815301,0.985581


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.05647222697734833, 'eval_accuracy': 0.9875164257555847, 'eval_f1_macro': 0.8153013845308229, 'eval_f1_weighted': 0.9855805282205194, 'eval_runtime': 51.0261, 'eval_samples_per_second': 29.828, 'eval_steps_per_second': 0.47, 'epoch': 2.0}


In [21]:

# =========================
# 7) Save final model
# =========================
SAVE_DIR = "./dept_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)
print("Labels:", labels[:10], "... total:", len(labels))


Saved to: ./dept_model
Labels: ['Administrative', 'Business Development', 'Consulting', 'Customer Support', 'Human Resources', 'Information Technology', 'Marketing', 'Other', 'Project Management', 'Purchasing'] ... total: 11


Testing on Test Data Set


In [30]:
Test_Data = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/Michi/df_profiles_cleansed"
df_test = pd.read_csv(Test_Data)

In [25]:
df_test.head()

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658


In [26]:
import pandas as pd
import numpy as np
import torch

from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Keep only rows where we have a true label
df_test = df_test.dropna(subset=["department"]).copy()
df_test["department"] = df_test["department"].astype(str).str.strip()


In [27]:
# =========================
# 2) Build / choose text input column
# =========================
# Preferred columns (adjust if you know your exact column)
candidate_cols = ["text", "position", "title", "headline", "job_title"]

text_col = next((c for c in candidate_cols if c in df_test.columns), None)

if text_col is None:
    # Fallback: build a text field from multiple columns if present
    parts = []
    for c in ["position", "organization", "company", "summary", "description"]:
        if c in df_test.columns:
            parts.append(df_test[c].fillna("").astype(str))
    if not parts:
        raise ValueError(
            f"Keine passende Textspalte gefunden. Spalten vorhanden: {list(df_test.columns)}. "
            "Bitte sag mir, welche Spalte als Input dienen soll (z.B. 'position')."
        )
    df_test["text_for_model"] = parts[0]
    for p in parts[1:]:
        df_test["text_for_model"] = df_test["text_for_model"] + " | " + p
    text_col = "text_for_model"

df_test[text_col] = df_test[text_col].fillna("").astype(str).str.strip()
df_test = df_test[df_test[text_col].str.len() > 0].copy()

print("Using text column:", text_col)
print("Rows for evaluation:", len(df_test))


Using text column: position
Rows for evaluation: 2615


In [28]:
# =========================
# 3) Load model + create pipeline
# =========================
# Path where you saved trainer.save_model(...)
# If you trained in the notebook, it was: SAVE_DIR = "./dept_model"
MODEL_DIR = "./dept_model"

clf = pipeline(
    "text-classification",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=32,   # match training setting (or 64 if you trained with 64)
)


The tokenizer you are loading from './dept_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cpu


In [31]:
# ========================
# 4) Predict in batches
# =========================
def predict_labels(texts, batch_size=64):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[i:i+batch_size]]
        out = clf(batch)  # returns [{'label': 'X', 'score': 0.99}, ...]
        preds.extend(out)
    return preds

preds = predict_labels(df_test[text_col].tolist(), batch_size=64)

df_test["pred_department"] = [p["label"] for p in preds]
df_test["pred_score"] = [float(p["score"]) for p in preds]


In [32]:
#
# =========================
# 5) Metrics
# =========================
y_true = df_test["department"].tolist()
y_pred = df_test["pred_department"].tolist()

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

print("\n=== TEST METRICS ===")
print("Accuracy:", round(acc, 4))
print("F1 macro:", round(f1_macro, 4))
print("F1 weighted:", round(f1_weighted, 4))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, digits=3))

# Optional: confusion matrix (labels order = sorted unique true labels)
labels = sorted(list(set(y_true) | set(y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=labels)

print("\nConfusion matrix labels order:")
print(labels)


=== TEST METRICS ===
Accuracy: 0.3426
F1 macro: 0.3444
F1 weighted: 0.3166

=== CLASSIFICATION REPORT ===
                        precision    recall  f1-score   support

        Administrative      0.052     0.310     0.089        84
  Business Development      0.361     0.449     0.400        78
            Consulting      0.435     0.605     0.506       195
      Customer Support      0.000     0.000     0.000        48
       Human Resources      0.333     0.232     0.274        69
Information Technology      0.336     0.676     0.449       309
             Marketing      0.335     0.436     0.379       133
                 Other      0.504     0.101     0.169      1235
    Project Management      0.390     0.705     0.502       173
            Purchasing      0.187     0.361     0.246        72
                 Sales      0.817     0.735     0.774       219

              accuracy                          0.343      2615
             macro avg      0.341     0.419     0.344      

In [37]:
mask = df_test["department"] != "Other"
print(classification_report(
    df_test.loc[mask, "department"],
    df_test.loc[mask, "pred_department"]
))


                        precision    recall  f1-score   support

        Administrative       0.22      0.31      0.26        84
  Business Development       0.49      0.45      0.47        78
            Consulting       0.73      0.61      0.66       195
      Customer Support       0.00      0.00      0.00        48
       Human Resources       0.64      0.23      0.34        69
Information Technology       0.65      0.68      0.66       309
             Marketing       0.66      0.44      0.52       133
                 Other       0.00      0.00      0.00         0
    Project Management       0.51      0.71      0.59       173
            Purchasing       0.53      0.36      0.43        72
                 Sales       0.92      0.74      0.82       219

              accuracy                           0.56      1380
             macro avg       0.49      0.41      0.43      1380
          weighted avg       0.62      0.56      0.58      1380



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
